# Plant Disease Detection - Reference Solution

Random Forest classifier on extracted leaf image features for 11-class disease detection.

In [ ]:
import csv, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

FEATURE_COLS = ['leaf_color_r','leaf_color_g','leaf_color_b','texture_contrast',
                'texture_homogeneity','lesion_area_pct','image_width','image_height','brightness']
CROP_ENCODE = {'tomato':0,'potato':1,'corn':2,'wheat':3,'grape':4}

def load(path):
    with open(path,'r',encoding='utf-8') as f: return list(csv.DictReader(f))

def to_X(rows):
    return np.array([[float(r[c]) for c in FEATURE_COLS] + [CROP_ENCODE.get(r.get('crop_type',''),-1)] for r in rows])

train = load('dataset/public/train.csv')
test = load('dataset/public/test.csv')
X_train, y_train = to_X(train), [r['disease'] for r in train]
X_test = to_X(test)
print(f'Train: {len(train)} | Test: {len(test)}')

In [ ]:
model = Pipeline([('sc', StandardScaler()), ('rf', RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=42))])
model.fit(X_train, y_train)
preds = model.predict(X_test)

with open('predictions.csv','w',newline='') as f:
    w = csv.DictWriter(f,fieldnames=['image_id','predicted_disease'])
    w.writeheader()
    for row,p in zip(test,preds): w.writerow({'image_id':row['image_id'],'predicted_disease':p})

print(f'Saved {len(preds)} predictions')